# Score every customer

*Data Analysis in Python · Topic 11.3*

Your CRM team is planning January's emails, and they want one list per segment. Score every customer of 2024 and 2025 on recency, frequency and monetary value as of January 1, 2026, and name their segments.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the cell that reads the `valid_orders` table: the orders before `CUTOFF` that weren't cancelled.
2. In the first cell marked ✏️, create the `rfm` table: one row per customer in the `valid_orders` table, with the customer IDs as the index and these columns:
   - `recency`: the whole days from the customer's last order to `SNAPSHOT`
   - `frequency`: the customer's orders
   - `monetary`: the sum of the customer's `subtotal`
3. In the second cell marked ✏️, add three scores to the `rfm` table, as whole numbers from 1 to 5:
   - `r_score`: the quintile of `recency`, with 5 for the most recent customers
   - `f_score`: 1 for one order, 2 for two, 3 for three, 4 for four or five and 5 for six or more
   - `m_score`: the quintile of `monetary`, with 5 for the customers who spent the most
4. In the last cell marked ✏️, write the function `rfm_segment(r_score, f_score)`, which returns the first of these segments that matches, and add its result to the `rfm` table as `segment`. Then create `segment_counts`: the customers in each segment.
   - `Champions`: an `r_score` of 4 or 5 and an `f_score` of 4 or 5
   - `Loyal`: an `f_score` of 3 or more
   - `New`: an `r_score` of 4 or 5 and an `f_score` of 1
   - `At risk`: an `r_score` of 1 or 2 and an `f_score` of 2 or more
   - `Lost`: an `r_score` of 1 or 2
   - `Promising`: everyone else
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `rfm` table has 41,776 customers, with $5,444,750.57 of spend. 8,494 customers have an `r_score` of 5, and 918 an `f_score` of 5. `segment_counts` has 6 segments, from 14,069 Lost customers to 1,448 Champions.

## Parameters

In [ ]:
import numpy as np
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
CUTOFF = "2026-01-01"  # The timestamps are in UTC: the orders before it are the orders of 2024 and 2025.
SNAPSHOT = CUTOFF

In [ ]:
# The orders of 2024 and 2025 that weren't cancelled. Run it, and leave it as it is.
orders = pd.read_csv(DATA_URL + "orders.csv", parse_dates=["created_at"],
                     usecols=["order_id", "customer_id", "status", "subtotal", "coupon", "created_at"])
orders = orders[orders["created_at"] < CUTOFF]
valid_orders = orders[orders["status"] != "Cancelled"]
valid_orders.shape

## Recency, frequency and monetary value

In [ ]:
rfm = valid_orders.groupby("customer_id").agg(last_order=("created_at", "max"), frequency=("order_id", "count"),
                                              monetary=("subtotal", "sum"))
rfm["recency"] = (pd.Timestamp(SNAPSHOT) - rfm["last_order"]).dt.days
rfm[["recency", "frequency", "monetary"]].describe().round(1)

## The scores

In [ ]:
rfm["r_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.cut(rfm["frequency"], bins=[0, 1, 2, 3, 5, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm[["r_score", "f_score", "m_score"]].apply(pd.Series.value_counts)

## The segments

In [ ]:
def rfm_segment(r_score, f_score):
    """Return the RFM segment of a customer's recency and frequency scores."""
    if r_score >= 4 and f_score >= 4:
        return "Champions"
    if f_score >= 3:
        return "Loyal"
    if r_score >= 4 and f_score == 1:
        return "New"
    if r_score <= 2 and f_score >= 2:
        return "At risk"
    if r_score <= 2:
        return "Lost"
    return "Promising"


def row_segment(row):
    """Return the RFM segment of one row of the rfm table."""
    return rfm_segment(row["r_score"], row["f_score"])


rfm["segment"] = rfm.apply(row_segment, axis=1)
segment_counts = rfm["segment"].value_counts()
segment_counts

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, calls your function and reads the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def counts(column):
        return {int(score): int(n) for score, n in rfm[column].value_counts().items()}

    rfm = found.get("rfm")
    columns = ["recency", "frequency", "monetary"]
    rfm_ok = False
    if not isinstance(rfm, pd.DataFrame) or [column for column in columns if column not in rfm.columns]:
        results.append((False, "rfm should be a DataFrame with the columns recency, frequency and monetary, one row per customer."))
    elif len(rfm) == 43509:
        results.append((False, "The rfm table has 43,509 customers: it counts cancelled orders. Use the valid_orders table."))
    elif len(rfm) != 41776:
        results.append((False, f"The rfm table should have one row per customer in the valid_orders table: 41,776. It has {len(rfm):,}."))
    elif int(rfm["frequency"].sum()) != 61702 or abs(rfm["monetary"].sum() - 5444750.57) > 0.01:
        results.append((False, f"frequency should count each customer's orders, 61,702 in all, and monetary sum their subtotal, $5,444,750.57. They add up to {rfm['frequency'].sum():,} and ${rfm['monetary'].sum():,.2f}."))
    elif not pd.api.types.is_integer_dtype(rfm["recency"]) or int(rfm["recency"].sum()) != 11304339:
        results.append((False, "recency should be the whole days from each customer's last order to SNAPSHOT: (pd.Timestamp(SNAPSHOT) - last_order).dt.days. The most recent customers have 0."))
    else:
        rfm_ok = True
        results.append((True, "41,776 customers, with 61,702 orders and $5,444,750.57 of spend."))

    if rfm_ok:
        scores = ["r_score", "f_score", "m_score"]
        if [score for score in scores if score not in rfm.columns]:
            results.append((False, "The rfm table should have the columns r_score, f_score and m_score."))
        elif not all(pd.api.types.is_integer_dtype(rfm[score]) for score in scores):
            results.append((False, "The scores should be whole numbers: add .astype(int) after pd.qcut() and pd.cut(), which return categories."))
        elif rfm.loc[rfm["recency"] < 30, "r_score"].mean() == 1:
            results.append((False, "The most recent customers have an r_score of 1. Fewer days is better: give recency's quintiles the labels [5, 4, 3, 2, 1]."))
        elif counts("r_score") != {1: 8327, 2: 8357, 3: 8341, 4: 8257, 5: 8494}:
            results.append((False, f"r_score should be the quintiles of recency, with 5 for the most recent: 8,494 customers have a 5. It has {counts('r_score')}."))
        elif counts("f_score") == {1: 8356, 2: 8355, 3: 8355, 4: 8355, 5: 8355}:
            results.append((False, "f_score splits the one-order customers into five equal groups by their row order, so 16,000 of them have a score of 3 or more. Use pd.cut() with the bins [0, 1, 2, 3, 5, np.inf]."))
        elif counts("f_score") != {1: 31107, 2: 7150, 3: 1508, 4: 1093, 5: 918}:
            results.append((False, f"f_score should be 1 for one order, 2 for two, 3 for three, 4 for four or five and 5 for six or more: pd.cut() with the bins [0, 1, 2, 3, 5, np.inf]. It has {counts('f_score')}."))
        elif counts("m_score") != {1: 8473, 2: 8892, 3: 7701, 4: 8355, 5: 8355}:
            results.append((False, f"m_score should be the quintiles of monetary, with 5 for the customers who spent the most. It has {counts('m_score')}."))
        else:
            results.append((True, "The scores: a fifth of the customers in each recency and monetary score, and 31,107 one-order customers with an f_score of 1."))

    segment = found.get("rfm_segment")
    cases = [((5, 5), "Champions"), ((3, 5), "Loyal"), ((5, 3), "Loyal"), ((1, 3), "Loyal"), ((4, 1), "New"), ((2, 2), "At risk"),
             ((1, 1), "Lost"), ((3, 1), "Promising"), ((4, 2), "Promising"), ((3, 2), "Promising")]
    if not callable(segment):
        results.append((False, "rfm_segment should be a function: def rfm_segment(r_score, f_score): …"))
    else:
        try:
            wrong = [(scores, name, segment(*scores)) for scores, name in cases if segment(*scores) != name]
        except Exception as error:
            wrong = [((5, 5), "Champions", f"an error, {type(error).__name__}")]
        if wrong:
            scores, name, got = wrong[0]
            results.append((False, f"rfm_segment({scores[0]}, {scores[1]}) should return {name!r}. It returns {got!r}. The rules run in the brief's order, and the first one that matches wins."))
        else:
            results.append((True, "rfm_segment() names every combination of scores the way the brief does."))

    expected = {"Lost": 14069, "Promising": 11437, "New": 10650, "At risk": 2101, "Loyal": 2071, "Champions": 1448}
    segment_counts = found.get("segment_counts")
    if rfm_ok and "segment" not in rfm.columns:
        results.append((False, "The rfm table should have a segment column: rfm_segment() of each row's r_score and f_score."))
    elif not isinstance(segment_counts, pd.Series):
        results.append((False, "segment_counts should be a Series: rfm[\"segment\"].value_counts()."))
    elif {str(name): int(n) for name, n in segment_counts.items()} != expected:
        results.append((False, f"segment_counts should be {', '.join(f'{name} {n:,}' for name, n in expected.items())}. It is {dict(segment_counts)}."))
    else:
        results.append((True, "14,069 Lost customers, and 1,448 Champions who ordered at least four times, most recently in the last 171 days."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
rfm = valid_orders.groupby("customer_id").agg(last_order=("created_at", "max"), frequency=("order_id", "count"),
                                              monetary=("subtotal", "sum"))
rfm["recency"] = (pd.Timestamp(SNAPSHOT) - rfm["last_order"]).dt.days
rfm[["recency", "frequency", "monetary"]].describe().round(1)
```

```python
rfm["r_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.cut(rfm["frequency"], bins=[0, 1, 2, 3, 5, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm[["r_score", "f_score", "m_score"]].apply(pd.Series.value_counts)
```

```python
def rfm_segment(r_score, f_score):
    """Return the RFM segment of a customer's recency and frequency scores."""
    if r_score >= 4 and f_score >= 4:
        return "Champions"
    if f_score >= 3:
        return "Loyal"
    if r_score >= 4 and f_score == 1:
        return "New"
    if r_score <= 2 and f_score >= 2:
        return "At risk"
    if r_score <= 2:
        return "Lost"
    return "Promising"


def row_segment(row):
    """Return the RFM segment of one row of the rfm table."""
    return rfm_segment(row["r_score"], row["f_score"])


rfm["segment"] = rfm.apply(row_segment, axis=1)
segment_counts = rfm["segment"].value_counts()
segment_counts
```

</details>